# **Retail Revenue Intelligence Platform**



## Notebook Outline

This notebook establishes the business context and provides an initial understanding of the dataset before any cleaning or analysis is performed.

*Notebook content* :

- Business Understanding
- Project Objective
- Business Questions
- Dataset Overview
- Dataset Structure
- Data Dictionary

## Business Understanding
A retail company has observed fluctuations in revenue and category performance throughout 2016 but lacks visibility into the underlying drivers of these changes. Leadership requires an analytics solution that not only reports business performance but also explains why performance changes occur and supports data-driven decision making. This project aims to build an end-to-end analytics workflow that transforms raw transaction data into actionable business insights.

## Project Objective

The objective of this project is to build an end-to-end Retail Revenue Intelligence Platform that transforms raw point-of-sale (POS) transaction data into actionable business insights. The project aims to establish a structured analytics workflow by cleaning and validating the data, designing a dimensional data model, performing SQL-based business analysis, defining key performance indicators (KPIs), and conducting a hypothesis-driven root cause analysis to identify the factors influencing revenue performance. The final deliverables include an executive dashboard and business recommendations to support informed decision-making.

## Business Questions
These are the questions our project must answer.

**Revenue**

How has revenue changed over time?

Which categories contribute the most to revenue?

Which products drive revenue growth?

**Customers**

How many active customers does the business have?

How frequently do customers make purchases?

Which customers contribute the highest revenue?

**Transactions**

What is the average basket value?

How many items are purchased per transaction?

**Investigation**

What caused the largest change in revenue during 2016?

Which business driver contributed the most to that change?

## Dataset Overview

The dataset contains retail point-of-sale (POS) transaction records collected throughout 2016. It consists of 131,706 transaction line items, representing approximately 64,700 customer transactions, 22,625 unique customers, 5,242 products (SKUs), and 187 product categories.

Each record captures information about a product purchased within a transaction, including the purchase date, customer, product, quantity purchased, and sales amount. The dataset supports customer, product, transaction, category, and time-based analysis.

In [ ]:
import pandas as pd

In [23]:
df = pd.read_csv("../data/raw/scanner_data.csv")

### Why this dataset?

This dataset is well suited for business analytics because it captures customer purchases at the transaction level, enabling analysis across multiple business dimensions including customers, products, categories, transactions, and time.

In [21]:
df.head()

,Unnamed: 0,Date,Customer_ID,Transaction_ID,SKU_Category,SKU,Quantity,Sales_Amount
0,1,02/01/2016,2547,1,X52,0EM7L,1.0,3.13
1,2,02/01/2016,822,2,2ML,68BRQ,1.0,5.46
2,3,02/01/2016,3686,3,0H2,CZUZX,1.0,6.35
3,4,02/01/2016,3719,4,0H2,549KK,1.0,5.59
4,5,02/01/2016,9200,5,0H2,K8EHH,1.0,6.88


In [16]:
df.shape

(131706, 8)

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131706 entries, 0 to 131705
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   Unnamed: 0      131706 non-null  int64  
 1   Date            131706 non-null  object 
 2   Customer_ID     131706 non-null  int64  
 3   Transaction_ID  131706 non-null  int64  
 4   SKU_Category    131706 non-null  object 
 5   SKU             131706 non-null  object 
 6   Quantity        131706 non-null  float64
 7   Sales_Amount    131706 non-null  float64
dtypes: float64(2), int64(3), object(3)
memory usage: 8.0+ MB


In [18]:
df.describe()

,Unnamed: 0,Customer_ID,Transaction_ID,Quantity,Sales_Amount
count,131706.000000,131706.000000,131706.000000,131706.000000,131706.000000
mean,65853.500000,12386.450367,32389.604187,1.485311,11.981524
std,38020.391614,6086.447552,18709.901238,3.872667,19.359699
min,1.000000,1.000000,1.000000,0.010000,0.020000
25%,32927.250000,7349.000000,16134.000000,1.000000,4.230000
50%,65853.500000,13496.000000,32620.000000,1.000000,6.920000
75%,98779.750000,17306.000000,48548.000000,1.000000,12.330000
max,131706.000000,22625.000000,64682.000000,400.000000,707.730000


In [19]:
df.columns

Index(['Unnamed: 0', 'Date', 'Customer_ID', 'Transaction_ID', 'SKU_Category',
       'SKU', 'Quantity', 'Sales_Amount'],
      dtype='object')

##  Dataset Structure

 *Dataset Shape*
- There are 131706 rows and 8 columns

- Each row represents one SKU (product) purchased within a retail transaction.

*Columns in the Dataset*
- There are 8 columns - Index(['Unnamed: 0', 'Date', 'Customer_ID', 'Transaction_ID', 'SKU_Category',
       'SKU', 'Quantity', 'Sales_Amount'],
      dtype='object')

*Data Types*
-  Column Value Distribution

   **Numerical**- Quantity, Sales_Amount

   **Categorical(identifiers)**- Customer_ID, Transaction_ID, SKU, SKU_Category

   **Date**- Date

   **Index**- Unnamed: 0


*Dataset Grain*
- Each row represents one SKU purchased within one retail transaction.

**example** -
<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Unnamed: 0</th>
      <th>Date</th>
      <th>Customer_ID</th>
      <th>Transaction_ID</th>
      <th>SKU_Category</th>
      <th>SKU</th>
      <th>Quantity</th>
      <th>Sales_Amount</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>1</td>
      <td>02/01/2016</td>
      <td>2547</td>
      <td>1</td>
      <td>X52</td>
      <td>0EM7L</td>
      <td>1.0</td>
      <td>3.13</td>
    </tr>
  </tbody>
</table>
</div>


in a single trip there could be multiple Transaction_ID's based on multiple SKUs

*Key Business Entities*
| Business Entity  | Column         |
| ---------------- | -------------- |
| Customer         | Customer_ID    |
| Transaction      | Transaction_ID |
| Product          | SKU            |
| Product Category | SKU_Category   |
| Time             | Date           |

*Relationships Between Entities*
- One customer can make multiple transactions over time.
- One transaction can contain multiple products (SKUs).
- Each product belongs to a single product category.
- Multiple transactions can occur on the same date.

*Dataset Summary*
| Attribute         | Observation                                                                   |
| ----------------- | ----------------------------------------------------------------------------- |
| Total Rows        | 131,706                                                                       |
| Total Columns     | 8                                                                             |
| Missing Values    | None (all columns have 131,706 non-null values)                               |
| Memory Usage      | ~8.0 MB                                                                       |
| Numerical Columns | 5 (`Unnamed: 0`, `Customer_ID`, `Transaction_ID`, `Quantity`, `Sales_Amount`) |
| Text Columns      | 3 (`Date`, `SKU_Category`, `SKU`)                                             |





## Data Dictionary
| Column         | Data Type | Description                        |
| -------------- | --------- | ---------------------------------- |
| Unnamed: 0     | Integer   | CSV row index                      |
| Date           | Object    | Transaction date                   |
| Customer_ID    | Integer   | Unique customer identifier         |
| Transaction_ID | Integer   | Unique shopping transaction        |
| SKU_Category   | Object    | Product category code              |
| SKU            | Object    | Product code                       |
| Quantity       | Float     | Number of units purchased          |
| Sales_Amount   | Float     | Revenue generated by the line item |

## Assumptions & Limitations

**Assumptions**
- Each Transaction_ID represents one shopping visit.
- Sales_Amount represents revenue.
- Each row represents one purchased SKU within a transaction.
**Limitations**
- No product names.
- No store/location information.
- No customer demographics.
- No cost or profit information.
- No inventory or stock data.

## Next Steps

Having established the business context and understood the dataset structure, the next phase will assess the quality of the data. This assessment will validate the dataset for missing values, duplicates, inconsistent data types, invalid values, and outliers before any transformations are performed.